# Lab 7: Customer Segmentation and Predictive Analytics

**Dataset:** UCI Online Retail (541,909 transactions)  
**Environment:** Python in Google Colab

> Note: the supplied PDF is named Lab 7 but its heading says Problem Statement 6. This notebook follows the customer-segmentation task in that PDF.

## Problem statement

Engineer RFM and purchasing features, segment customers using K-Means and hierarchical clustering, validate and visualize the clusters, build Random Forest and SVM classifiers for high-value membership, and propose segment-specific marketing strategies.

In [ ]:
# Colab setup
!pip -q install openpyxl seaborn plotly scikit-learn

import io, requests, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, silhouette_score,
                             adjusted_rand_score, roc_curve)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

RANDOM_STATE = 42
FEATURES = ['Recency','Frequency','Monetary','AverageTransactionValue',
            'QuantityPurchased','PurchaseFrequency']
sns.set_theme(style='whitegrid')

In [ ]:
# Download the official UCI dataset
url = 'https://archive.ics.uci.edu/static/public/352/online+retail.zip'
content = requests.get(url, timeout=120).content
with zipfile.ZipFile(io.BytesIO(content)) as z:
    with z.open('Online Retail.xlsx') as f:
        raw = pd.read_excel(f, engine='openpyxl')
print(raw.shape)
raw.head()

## Preprocessing and customer-level feature engineering

Remove missing customer IDs, cancellations, non-positive quantities/prices, then engineer six customer features. IQR capping limits extreme influence; `log1p` and standardization prepare skewed monetary/count features for distance-based clustering.

In [ ]:
raw['InvoiceDate'] = pd.to_datetime(raw['InvoiceDate'])
clean = raw.dropna(subset=['CustomerID','InvoiceDate']).copy()
clean = clean[~clean['InvoiceNo'].astype(str).str.upper().str.startswith('C')]
clean = clean[(clean['Quantity'] > 0) & (clean['UnitPrice'] > 0)].copy()
clean['CustomerID'] = clean['CustomerID'].astype(int).astype(str)
clean['TotalPrice'] = clean['Quantity'] * clean['UnitPrice']
snapshot = clean['InvoiceDate'].max() + pd.Timedelta(days=1)
span = clean.groupby('CustomerID')['InvoiceDate'].agg(lambda x: max((x.max()-x.min()).days,1))
customers = clean.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot-x.max()).days),
    Frequency=('InvoiceNo','nunique'), Monetary=('TotalPrice','sum'),
    AverageTransactionValue=('TotalPrice','mean'), QuantityPurchased=('Quantity','sum'),
    Country=('Country', lambda x: x.mode().iloc[0])).reset_index()
customers['PurchaseFrequency'] = customers['Frequency'] / customers['CustomerID'].map(span) * 30
model_features = customers[FEATURES].copy()
for c in FEATURES:
    q1,q3 = model_features[c].quantile([.25,.75]); iqr=q3-q1
    model_features[c] = model_features[c].clip(max(0,q1-1.5*iqr), q3+1.5*iqr)
scaled = StandardScaler().fit_transform(np.log1p(model_features))
print('Raw:',len(raw),'Clean:',len(clean),'Customers:',len(customers))
customers.describe().T

## K-Means: elbow and silhouette selection

In [ ]:
rows=[]
for k in range(2,9):
    km=KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20).fit(scaled)
    rows.append([k,km.inertia_,silhouette_score(scaled,km.labels_)])
scores=pd.DataFrame(rows,columns=['k','WCSS','Silhouette'])
selected_k=int(scores.loc[scores.Silhouette.idxmax(),'k'])
fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].plot(scores.k,scores.WCSS,'o-'); ax[0].set(title='Elbow Method',xlabel='k',ylabel='WCSS')
ax[1].plot(scores.k,scores.Silhouette,'o-'); ax[1].set(title='Silhouette Scores',xlabel='k',ylabel='Score')
plt.show(); scores

**Generated result:** the reproducible local run selected **k = 2** with silhouette score **0.379**.

## K-Means, hierarchical clustering, and dendrogram

In [ ]:
kmeans=KMeans(n_clusters=selected_k,random_state=RANDOM_STATE,n_init=30)
customers['KMeansCluster']=kmeans.fit_predict(scaled)
hier=AgglomerativeClustering(n_clusters=selected_k,linkage='ward')
customers['HierarchicalCluster']=hier.fit_predict(scaled)
print('K-Means silhouette:',silhouette_score(scaled,customers.KMeansCluster))
print('Hierarchical silhouette:',silhouette_score(scaled,customers.HierarchicalCluster))
print('Adjusted Rand agreement:',adjusted_rand_score(customers.KMeansCluster,customers.HierarchicalCluster))
idx=np.random.default_rng(RANDOM_STATE).choice(len(customers),500,replace=False)
plt.figure(figsize=(14,5)); dendrogram(linkage(scaled[idx],method='ward'),truncate_mode='lastp',p=40)
plt.title('Hierarchical Clustering Dendrogram'); plt.ylabel('Ward distance'); plt.show()

## PCA visualizations and customer profiling

In [ ]:
pca=PCA(n_components=3,random_state=RANDOM_STATE)
customers[['PCA1','PCA2','PCA3']]=pca.fit_transform(scaled)
px.scatter(customers,x='PCA1',y='PCA2',color=customers.KMeansCluster.astype(str),
           hover_data=['CustomerID','Recency','Frequency','Monetary'],title='2D PCA Customer Segments').show()
px.scatter_3d(customers,x='PCA1',y='PCA2',z='PCA3',color=customers.KMeansCluster.astype(str),
              hover_data=['CustomerID','Recency','Frequency','Monetary'],title='Interactive 3D Customer Segments').show()

In [ ]:
profile=customers.groupby('KMeansCluster').agg(
    Customers=('CustomerID','count'),Mean_Recency=('Recency','mean'),
    Mean_Frequency=('Frequency','mean'),Mean_Monetary=('Monetary','mean'),
    Mean_AverageTransactionValue=('AverageTransactionValue','mean'),
    Mean_QuantityPurchased=('QuantityPurchased','mean'),
    Mean_PurchaseFrequency=('PurchaseFrequency','mean')).round(2)
high_cluster=int(profile.Mean_Monetary.idxmax())
customers['HighValue']=(customers.KMeansCluster==high_cluster).astype(int)
profile

## Random Forest and SVM high-value prediction

In [ ]:
X=customers[FEATURES]; y=customers.HighValue
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=RANDOM_STATE,stratify=y)
models={
'Random Forest':Pipeline([('scale',StandardScaler()),('model',RandomForestClassifier(n_estimators=350,class_weight='balanced',min_samples_leaf=2,random_state=RANDOM_STATE))]),
'SVM':Pipeline([('scale',StandardScaler()),('model',SVC(probability=True,class_weight='balanced',random_state=RANDOM_STATE))])}
rows=[]
fig,axes=plt.subplots(1,2,figsize=(9,4))
for ax,(name,model) in zip(axes,models.items()):
    model.fit(X_train,y_train); pred=model.predict(X_test); prob=model.predict_proba(X_test)[:,1]
    rows.append([name,accuracy_score(y_test,pred),precision_score(y_test,pred),recall_score(y_test,pred),f1_score(y_test,pred),roc_auc_score(y_test,prob)])
    sns.heatmap(confusion_matrix(y_test,pred),annot=True,fmt='d',cbar=False,ax=ax); ax.set_title(name)
plt.show()
metrics=pd.DataFrame(rows,columns=['Model','Accuracy','Precision','Recall','F1','ROC_AUC'])
metrics

**Generated result:** the best local-run ROC-AUC was **0.999** from **Random Forest**. Accuracy alone is not used because the high-value segment is smaller than the remaining population.

## Feature importance

In [ ]:
importance=pd.Series(models['Random Forest'].named_steps['model'].feature_importances_,index=FEATURES).sort_values()
importance.plot.barh(title='Random Forest Feature Importance'); plt.show(); importance.sort_values(ascending=False)

## Marketing recommendations and conclusion

- **Champions:** VIP access, premium bundles, referrals, and early launches.
- **Loyal customers:** loyalty tiers, cross-sell, and replenishment reminders.
- **At-risk valuable customers:** personalized win-back campaigns and limited incentives.
- **Developing customers:** onboarding, starter bundles, and second-purchase coupons.

The workflow delivers validated customer segments and effective classifiers for high-value membership. These observational results support targeting decisions but do not establish that a campaign causes improved purchasing. Future work should validate recommendations through controlled A/B tests.

## Reproducibility and source

UCI Machine Learning Repository, **Online Retail**, DOI: `10.24432/C5BW33`. Use **Runtime > Run all** in Google Colab, then **File > Print > Save as PDF** for the official notebook export.